In [ ]:
# Install seluruh dependency dengan versi terpin di requirements.txt
%pip install -q -r requirements.txt


In [ ]:
# import library
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

## 1. Data acquisition dan loading

Dataset kompetisi tersedia di `dataset/`. Metadata publik untuk 20 judul yang tidak tercantum pada katalog lokal tersimpan di `external_data/missing_movies_metadata.csv`; sumber, endpoint, tanggal pengambilan, atribusi, dan audit kecocokan judul dicatat di `external_data/`. Akuisisi ulang dapat dilakukan melalui `scripts/build_pipeline.py --fetch-metadata` dengan variabel TMDB di `.env`.

In [ ]:

DATASET_DIR = Path("dataset")
SUPPORTED_EXTENSIONS = {".csv"}

datasets = {}

for file_path in sorted(DATASET_DIR.iterdir()):
    if not file_path.is_file() or file_path.suffix.lower() not in SUPPORTED_EXTENSIONS:
        continue

    try:
        extension = file_path.suffix.lower()
        if extension == ".csv":
            data = pd.read_csv(file_path)
        elif extension in {".xlsx", ".xls"}:
            data = pd.read_excel(file_path)
        elif extension == ".json":
            data = pd.read_json(file_path)
        else:
            data = pd.read_parquet(file_path)

        datasets[file_path.stem] = data
        print(f"\n{'=' * 80}\nDataset: {file_path.name}")
        print(f"Ukuran: {data.shape[0]:,} baris x {data.shape[1]:,} kolom")
        print("\nTipe data:")
        print(data.dtypes)
        print("\nMissing values:")
        missing = data.isna().sum().sort_values(ascending=False)
        print(missing[missing > 0] if missing.any() else "Tidak ada missing values")
        print("\nDuplikat:")
        print(data.duplicated().sum())
        print("\nStatistik numerik:")
        display(data.describe().T)
        print("\nLima baris pertama:")
        display(data.head())
    except Exception as error:
        print(f"Gagal memuat {file_path.name}: {error}")

print(f"\nTotal dataset berhasil dimuat: {len(datasets)}")
print("Nama dataset:", list(datasets.keys()))

## 2. Preprocessing dan pemeriksaan kualitas

Cell berikut memeriksa konsistensi kunci, tanggal, dan kelengkapan antar-sumber sebelum analisis.

data yg missing sangat sedikit, termasuk lengkap.

## 3. Exploratory data analysis

Analisis berikut meninjau skala target, kalender/libur, karakter cluster, harga, genre, serta anomali historis.

In [ ]:
# Validasi key dan kelengkapan data antar dataset
train = datasets["train"].copy()
test_history = datasets["test_history"].copy()
test = datasets["test"].copy()
movies = datasets["movies"].copy()
ticket_prices = datasets["ticket_prices"].copy()
holidays = datasets["holidays"].copy()

for data, date_column in [
    (train, "date_show"),
    (test_history, "date_show"),
    (test, "date_show"),
    (holidays, "date"),
]:
    data[date_column] = pd.to_datetime(data[date_column])

# 1. Cinema cluster: histori vs periode test
history_cinema_ids = set(train["cinema_ids"]) | set(test_history["cinema_ids"])
test_cinema_ids = set(test["cinema_ids"])
print("Cinema IDs hanya muncul di histori:", len(history_cinema_ids - test_cinema_ids))
print("Cinema IDs hanya muncul di test:", len(test_cinema_ids - history_cinema_ids))
print("Cinema IDs muncul di keduanya:", len(history_cinema_ids & test_cinema_ids))

# 2. Movie title: exact match dan kandidat setelah normalisasi sederhana
normalize_title = lambda series: (
    series.astype("string")
    .str.upper()
    .str.replace(r"\\(\\s*(2D|3D|IMAX|DUBBING)\\s*\\)", "", regex=True)
    .str.replace(r"\\b(2D|3D|IMAX|DUBBING)\\b", "", regex=True)
    .str.replace(r"[^A-Z0-9]+", " ", regex=True)
    .str.replace(r"\\s+", " ", regex=True)
    .str.strip()
)

movie_titles = set(movies["original_title"].dropna())
test_titles = set(test["movie_title"].dropna())
missing_exact_titles = sorted(test_titles - movie_titles)
print("\\nMovie title test yang tidak exact-match ke movies.csv:", len(missing_exact_titles))
print(missing_exact_titles[:20])

normalized_movie_titles = set(normalize_title(movies["original_title"].dropna()))
normalized_test_titles = normalize_title(pd.Series(missing_exact_titles))
resolved_after_normalization = set(normalized_test_titles) & normalized_movie_titles
print("Kandidat yang match setelah normalisasi format/spasi:", len(resolved_after_normalization))
print("Masih tidak ditemukan setelah normalisasi:", len(set(normalized_test_titles) - normalized_movie_titles))

# 3. Konsistensi nama kota
city_sources = {
    "train": train["city_name"],
    "test": test["city_name"],
    "ticket_prices": ticket_prices["city_name"],
}
normalized_city_sets = {
    name: set(values.dropna().astype("string").str.upper().str.strip())
    for name, values in city_sources.items()
}
all_cities = set.union(*normalized_city_sets.values())
city_comparison = pd.DataFrame({
    name: sorted(all_cities).copy() for name in normalized_city_sets
})
print("\\nJumlah kota unik:")
print({name: len(values) for name, values in normalized_city_sets.items()})
print("Kota yang ada di train tetapi tidak di ticket_prices:", sorted(normalized_city_sets["train"] - normalized_city_sets["ticket_prices"]))
print("Kota yang ada di test tetapi tidak di ticket_prices:", sorted(normalized_city_sets["test"] - normalized_city_sets["ticket_prices"]))

# 4. Setiap pasangan film-klaster di test harus memiliki 7 hari
pair_counts = test.groupby(["movie_title", "cinema_ids"]).size()
print("\\nDistribusi jumlah baris per pasangan film-klaster:")
print(pair_counts.value_counts().sort_index())
invalid_pairs = pair_counts[pair_counts != 7]
print(f"Pasangan yang bukan 7 baris: {len(invalid_pairs)}")
display(invalid_pairs.head(20).rename("jumlah_baris").to_frame())

In [ ]:
# Distribusi total_ticket dan skala MASE dari tiga hari histori pertama
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(train["total_ticket"], bins=60, color="steelblue", edgecolor="white")
axes[0].set_title("Distribusi total_ticket")
axes[0].set_xlabel("Total ticket")
axes[0].set_ylabel("Frekuensi")
axes[1].hist(np.log1p(train["total_ticket"]), bins=60, color="darkorange", edgecolor="white")
axes[1].set_title("Distribusi log1p(total_ticket)")
axes[1].set_xlabel("log1p(total_ticket)")
axes[1].set_ylabel("Frekuensi")
plt.tight_layout()
plt.show()


def compute_history_stats(history_df, n_days=3, id_cols=("movie_title", "cinema_ids")):
    """Hitung statistik agregat dari n hari PERTAMA observasi per pasangan
    (default: D1-D3, sesuai definisi hitung_skala pada deskripsi kompetisi).

    Diambil dengan cara: urutkan ascending berdasarkan date_show, lalu ambil
    n baris pertama tiap grup. Fungsi ini dipakai ulang di cell lain (lihat
    cell anomali di bawah) supaya D1-D3 tidak dihitung dengan logika berbeda
    di beberapa tempat.
    """
    id_cols = list(id_cols)
    sorted_df = history_df.sort_values(id_cols + ["date_show"])
    window = sorted_df.groupby(id_cols, group_keys=False).head(n_days)

    stats = window.groupby(id_cols, as_index=False).agg(
        mean_ticket=("total_ticket", "mean"),
        max_ticket=("total_ticket", "max"),
        min_ticket=("total_ticket", "min"),
        mean_total_show=("total_show", "mean"),
        mean_occupation_rate=("occupation_rate", "mean"),
        n_obs=("total_ticket", "size"),
    )
    return stats


# hitung_skala: rata-rata D1-D3 per pasangan film-klaster (mengikuti definisi resmi di deskripsi).
scale_table = compute_history_stats(test_history, n_days=3).rename(
    columns={"mean_ticket": "scale_raw"}
)
scale_table["scale"] = scale_table["scale_raw"].clip(lower=1)
scale_table["was_clipped"] = scale_table["scale_raw"] < 1
scale_table["at_floor"] = scale_table["scale_raw"] <= 1

print("Ringkasan scale_raw:")
display(scale_table["scale_raw"].describe().to_frame())
print(f"Pasangan yang benar-benar di-clip oleh max(1, scale): {scale_table['was_clipped'].sum():,} "
      f"({scale_table['was_clipped'].mean() * 100:.2f}%)")
print(f"Pasangan yang berada di floor scale=1: {scale_table['at_floor'].sum():,} "
      f"({scale_table['at_floor'].mean() * 100:.2f}%)")
print("Lima pasangan dengan scale terkecil:")
display(scale_table.nsmallest(5, "scale_raw"))

plt.figure(figsize=(9, 5))
plt.hist(np.log1p(scale_table["scale_raw"]), bins=50, color="seagreen", edgecolor="white")
plt.title("Distribusi log1p(scale) per pasangan film-klaster")
plt.xlabel("log1p(scale_raw)")
plt.ylabel("Jumlah pasangan")
plt.show()


In [ ]:
# Dampak hari libur dan efek H-1/H+1
import matplotlib.pyplot as plt

train_holiday = train.merge(
    holidays[["date", "day_tipe", "holiday_tipe", "holiday_name"]],
    left_on="date_show",
    right_on="date",
    how="left",
)

print("Rata-rata total_ticket berdasarkan tipe hari:")
day_summary = (
    train_holiday.groupby("day_tipe", dropna=False)["total_ticket"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)
display(day_summary)

print("Rata-rata total_ticket berdasarkan normal vs holiday:")
holiday_summary = (
    train_holiday.groupby("holiday_tipe", dropna=False)["total_ticket"]
    .agg(["count", "mean", "median"])
    .sort_values("mean", ascending=False)
)
display(holiday_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
day_summary["mean"].plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Rata-rata total_ticket berdasarkan tipe hari")
axes[0].set_xlabel("Tipe hari")
axes[0].set_ylabel("Rata-rata total_ticket")
axes[0].tick_params(axis="x", rotation=0)

holiday_summary["mean"].plot(kind="bar", ax=axes[1], color=["seagreen", "darkorange"], edgecolor="black")
axes[1].set_title("Rata-rata total_ticket: normal vs holiday")
axes[1].set_xlabel("Tipe tanggal")
axes[1].set_ylabel("Rata-rata total_ticket")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

major_holidays = (
    holidays.loc[holidays["holiday_tipe"] == "holiday", ["date", "holiday_name"]]
    .drop_duplicates()
    .sort_values("date")
)
print(f"Jumlah hari libur terdeteksi: {len(major_holidays)}")
display(major_holidays)

daily_ticket = train.groupby("date_show", as_index=False)["total_ticket"].mean()

holiday_effect_rows = []
for _, event in major_holidays.iterrows():
    for offset in [-1, 0, 1]:
        event_date = event["date"] + pd.Timedelta(days=offset)
        matching = daily_ticket.loc[daily_ticket["date_show"] == event_date, "total_ticket"]
        holiday_effect_rows.append({
            "holiday_date": event["date"],
            "holiday_name": event["holiday_name"],
            "offset": offset,
            "date_show": event_date,
            "mean_total_ticket": matching.iloc[0] if len(matching) else np.nan,
        })

holiday_effect = pd.DataFrame(holiday_effect_rows)
print("Efek H-1, hari-H, dan H+1 untuk setiap hari libur:")
display(holiday_effect)
print("Rata-rata efek berdasarkan offset (gabungan semua hari libur):")
offset_summary = holiday_effect.groupby("offset")["mean_total_ticket"].agg(["count", "mean", "median"])
display(offset_summary)

# Catatan: banyak tanggal libur (holidays.csv sampai Apr 2026) berada di luar
# rentang train.csv (Apr-Sep 2025), jadi mean_total_ticket-nya otomatis NaN.
# Baris NaN itu wajar dan diabaikan saat plotting di bawah.
plt.figure(figsize=(11, 5))
for holiday_name, event_data in holiday_effect.groupby("holiday_name"):
    event_data = event_data.sort_values("offset")
    if event_data["mean_total_ticket"].isna().all():
        continue
    plt.plot(
        event_data["offset"],
        event_data["mean_total_ticket"],
        marker="o",
        linewidth=1.5,
        alpha=0.8,
        label=holiday_name,
    )
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xticks([-1, 0, 1], ["H-1", "Hari H", "H+1"])
plt.xlabel("Posisi terhadap hari libur")
plt.ylabel("Rata-rata total_ticket")
plt.title("Efek hari libur terhadap penjualan tiket (semua holiday_tipe == \'holiday\')")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, ncol=1)
plt.tight_layout()
plt.show()


In [ ]:
# Perbandingan karakter antar cinema cluster
cluster_summary = (
    train.groupby("cinema_ids")
    .agg(
        mean_total_ticket=("total_ticket", "mean"),
        median_total_ticket=("total_ticket", "median"),
        mean_occupation_rate=("occupation_rate", "mean"),
        mean_total_show=("total_show", "mean"),
        observations=("total_ticket", "size"),
    )
    .sort_values("mean_total_ticket", ascending=False)
)
print("Distribusi skala klaster berdasarkan histori:")
display(cluster_summary.describe().T)
print("Klaster terbesar dan terkecil:")
display(pd.concat([cluster_summary.head(10), cluster_summary.tail(10)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(
    cluster_summary["mean_total_show"],
    cluster_summary["mean_occupation_rate"],
    alpha=0.65,
    color="teal",
)
axes[0].set_title("Total show vs occupation rate")
axes[0].set_xlabel("Rata-rata total_show")
axes[0].set_ylabel("Rata-rata occupation_rate (%)")
axes[1].scatter(
    cluster_summary["mean_total_show"],
    cluster_summary["mean_total_ticket"],
    alpha=0.65,
    color="crimson",
)
axes[1].set_title("Total show vs total ticket")
axes[1].set_xlabel("Rata-rata total_show")
axes[1].set_ylabel("Rata-rata total_ticket")
plt.tight_layout()
plt.show()

# Reverse-engineer estimasi kapasitas kursi per show.
train["estimated_capacity_per_show"] = np.where(
    (train["total_show"] > 0) & (train["occupation_rate"] > 0),
    train["total_ticket"] / (train["total_show"] * train["occupation_rate"] / 100),
    np.nan,
)
capacity_by_cluster = (
    train.groupby("cinema_ids")["estimated_capacity_per_show"]
    .agg(["count", "median", "mean", "min", "max"])
    .sort_values("median", ascending=False)
)
print("Estimasi kapasitas kursi per show per klaster:")
display(capacity_by_cluster.describe())
display(capacity_by_cluster.head(10))

In [ ]:
# Join harga tiket berdasarkan kota dan tipe hari
# 1. Join train dengan holidays untuk dapat day_tipe
train_h = train.merge(
    holidays.rename(columns={"date": "date_show"}),
    on="date_show",
    how="left",
)

# 2. Baru join dengan ticket_prices. Ini many-to-many by city_name karena tiap
#    kota punya 3 baris harga (Weekday/Friday/Weekend), makanya difilter oleh
#    price_day_normalized di bawah sebelum groupby, supaya tidak ada duplikasi baris.
train_prices = train_h.merge(
    ticket_prices,
    on="city_name",
    how="left",
    suffixes=("", "_price"),
)

train_prices["price_day_normalized"] = train_prices["day_tipe"].map({
    "weekday": "Weekday",
    "friday": "Friday",
    "weekend": "Weekend",
})

train_prices["city_day_price"] = np.where(
    train_prices["price_day"] == train_prices["price_day_normalized"],
    train_prices["ceil"],
    np.nan,
)

print("Baris tanpa harga kota-hari yang cocok:", train_prices["city_day_price"].isna().sum())

# Hapus baris duplikat hasil merge many-to-many, sisakan 1 baris harga per observasi asli
train_prices_clean = train_prices.dropna(subset=["city_day_price"])
print("Jumlah baris asli (train):", len(train), "| setelah join & filter:", len(train_prices_clean))

city_price_demand = (
    train_prices_clean.groupby("city_name")
    .agg(
        mean_price=("city_day_price", "mean"),
        mean_total_ticket=("total_ticket", "mean"),
        median_total_ticket=("total_ticket", "median"),
        mean_occupation_rate=("occupation_rate", "mean"),
        observations=("total_ticket", "size"),
    )
    .dropna()
)

print("Ringkasan harga tiket vs permintaan per kota:")
display(city_price_demand.sort_values("mean_price", ascending=False))

print("Korelasi harga rata-rata dengan permintaan kota:")
display(city_price_demand.corr(numeric_only=True))

plt.figure(figsize=(8, 5))
plt.scatter(
    city_price_demand["mean_price"],
    city_price_demand["mean_total_ticket"],
    alpha=0.7,
    color="purple",
)
for city, row in city_price_demand.iterrows():
    plt.annotate(city, (row["mean_price"], row["mean_total_ticket"]), fontsize=8, alpha=0.7)
plt.xlabel("Harga tiket rata-rata (Rp)")
plt.ylabel("Rata-rata total_ticket")
plt.title("Harga tiket vs permintaan per kota")
plt.tight_layout()
plt.show()


In [ ]:
# Analisis genre, decay historis, dan normalisasi title format
movie_lookup = movies[["original_title", "genre"]].drop_duplicates("original_title")
train_movies = train.merge(
    movie_lookup,
    left_on="movie_title",
    right_on="original_title",
    how="left",
)
train_movies["date_rank"] = train_movies.groupby(
    ["movie_title", "cinema_ids"]
)["date_show"].rank(method="dense", ascending=True)

# Bandingkan D1 dengan hari-hari berikutnya untuk pasangan yang punya histori cukup.
pair_first_last = (
    train_movies[train_movies["date_rank"] <= 7]
    .groupby(["movie_title", "cinema_ids", "genre", "date_rank"], dropna=False)["total_ticket"]
    .mean()
    .reset_index()
)
pair_decay = pair_first_last.pivot_table(
    index=["movie_title", "cinema_ids", "genre"],
    columns="date_rank",
    values="total_ticket",
)
pair_decay["decay_d7_vs_d1"] = pair_decay.get(7, np.nan) / pair_decay.get(1, np.nan)
print("Decay median berdasarkan genre (nilai < 1 berarti turun):")
display(
    pair_decay.reset_index()
    .groupby("genre", dropna=False)["decay_d7_vs_d1"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
    .head(30)
)

format_pattern = r"2D|3D|IMAX|DUBBING"
test["has_format_marker"] = test["movie_title"].str.contains(
    format_pattern, case=False, regex=True, na=False
)
format_titles = test.loc[test["has_format_marker"], "movie_title"].drop_duplicates().sort_values()
print(f"Jumlah title test dengan penanda format: {len(format_titles)}")
display(format_titles.to_frame("movie_title").head(50))
print("Jumlah title katalog yang menjadi duplikat setelah normalisasi format:")
movie_normalized = normalize_title(movies["original_title"])
normalized_duplicates = movies.assign(normalized_title=movie_normalized).groupby(
    "normalized_title"
)["original_title"].nunique()
display(normalized_duplicates[normalized_duplicates > 1].sort_values(ascending=False).head(30).to_frame("jumlah_title"))

In [ ]:
# Pemeriksaan representasi ekstrem dan inkonsistensi numerik
# FIX: cell ini sebelumnya menghitung ulang D1-D3 dengan logika terpisah dari cell
# scale/MASE di atas (duplikasi). Sekarang pakai ulang compute_history_stats()
# yang sudah didefinisikan sebelumnya, supaya D1-D3 hanya dihitung dengan satu
# definisi yang konsisten di seluruh notebook.
pair_d1_d3 = compute_history_stats(test_history, n_days=3).rename(
    columns={"mean_ticket": "mean_d1_d3", "max_ticket": "max_d1_d3"}
)

print("Pasangan dengan D1-D3 tertinggi:")
display(pair_d1_d3.sort_values("mean_d1_d3", ascending=False).head(20))
print("Pasangan dengan D1-D3 terendah:")
display(pair_d1_d3.sort_values("mean_d1_d3", ascending=True).head(20))

zero_show_positive_ticket = train[(train["total_show"] == 0) & (train["total_ticket"] > 0)]
positive_show_zero_ticket = train[(train["total_show"] > 0) & (train["total_ticket"] == 0)]
print("\nAnomali total_show=0 tetapi total_ticket>0:", len(zero_show_positive_ticket))
print("Anomali total_show>0 tetapi total_ticket=0:", len(positive_show_zero_ticket))

invalid_occupation = train[
    (train["occupation_rate"] < 0) | (train["occupation_rate"] > 100)
]
print("occupation_rate di luar rentang 0-100:", len(invalid_occupation))
print("Ringkasan nilai ekstrem total_ticket:")
display(train["total_ticket"].describe(percentiles=[.01, .05, .5, .95, .99]).to_frame())


## 4. Feature engineering, modeling, dan evaluation

Pipeline memakai ulang `compute_history_stats()` dari notebook untuk skala D1-D3, membangun fitur target D4-D10, membuat holdout temporal per pasangan, membandingkan baseline dengan LightGBM L1, dan menyimpan feature table. MASE ini adalah validasi lokal pada `train.csv`, bukan skor submission.

In [ ]:
from pathlib import Path
import subprocess, sys

project_root = Path.cwd()
subprocess.run([sys.executable, str(project_root / "scripts" / "build_pipeline.py")], check=True)
